In [ ]:
import numpy as np
import pandas as pd
from nltk.classify import NaiveBayesClassifier
from nltk.corpus import subjectivity
from nltk.sentiment import SentimentAnalyzer
from nltk.sentiment.util import *

import numpy as np 
import pandas as pd 
#pd.set_option('display.max_colwidth', -1)
from time import time
import re
import string
import os
#import emoji
from pprint import pprint
import collections

import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="darkgrid")
sns.set(font_scale=1.3)

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.metrics import classification_report

from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
#from sklearn.externals import joblib
import joblib as jb

#import gensim

from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize

import warnings
warnings.filterwarnings('ignore')

np.random.seed(37)

In [ ]:
#import stata_setup
#stata_setup.config("C:/Program Files/Stata17", "mp")

In [ ]:
# Reading the saved data pickle file
df_vix = pd.read_pickle('/Users/Farid Srouji/Documents/ProjectOptionsPredictions/Data/PKL/financialmodelingprep/pickled_one_year_filtered_lead_para_sp500_clean.pkl')

In [ ]:
df_vix

In [ ]:
#df_vix.columns = ['date', 'close', 'adj close', 'articles']
#df_vix[['date', 'close']] = df_vix['close'].split(' ', 1, expand=True)
#df_vix = pd.DataFrame(df_vix.close.first.split(' ',1).tolist(),
                                 #columns = ['date','close'])
#print(df_vix.close. to_string().split(' ',1))
#df_new = pd.DataFrame(df_vix.close.to_string().split(' ',1), columns = ['date','close'])

In [ ]:
df_vix

In [ ]:
#from pandas import DataFrame

#df1 = DataFrame(df_vix[df_vix.columns[0]].to_string().split(" ",569), columns=['date'])
ts=pd.Series(df_vix[df_vix.columns[0]], index=pd.date_range("25/09/2019", periods=569))
ts.plot()

In [ ]:
df_vix = df_vix.reset_index()
df_vix['VIX'] = df_vix['adj close'].apply(np.int64)
#df_vix['text'] = df_vix['articles'].str()

In [ ]:
# selecting the VIX and articles
df_vix = df_vix[['VIX', 'articles']]

In [ ]:
#df_vix['articles'] = df_vix['articles'].map(lambda x: x.lstrip('.-'))

In [ ]:
df_vix

In [ ]:
df = df_vix[['VIX']].copy()
df


In [ ]:
df_vix = df_vix.reindex(np.random.permutation(df_vix.index))  
df_vix = df_vix[['VIX', 'articles']]

In [ ]:
# Adding new columns to the data frame
df['compound'] = ''
df['neg'] = ''
df['neu'] = ''
df['pos'] = ''

In [ ]:
df

In [ ]:
#import nltk
#nltk.download('vader_lexicon')

In [ ]:
from nltk.sentiment.vader import SentimentIntensityAnalyzer
import unicodedata
sid = SentimentIntensityAnalyzer()
countNeg=0
countNeu=0
countPos=0
#print(df_vix['articles'])
for date, row in df_vix.T.iteritems():
    print("The date is:"+str(date))
    try:
        sentence = unicodedata.normalize('NFKD', df_vix.loc[date, 'articles'])#.encode('ascii','ignore').decode('ascii')
        #sentence = unicodedata.normalize('NFKD', df_vix.loc[date, 'articles']).encode('ascii','ignore')
        # add my own sentence Options sentiment.
        #print("#1")
        #print (df_vix.loc[date, 'articles'])
        ss = sid.polarity_scores(sentence)
        #print(sentence)
        #print("#2")
        df.at[date, 'compound'] = ss['compound']
        df.at[date, 'neg'] = ss['neg']
        df.at[date, 'neu'] = ss['neu']
        df.at[date, 'pos'] = ss['pos']
        if ((ss['pos'] + ss['neg']) != 0):
            df.at[date, 'ForGraph'] = 100*(ss['pos'] / (ss['pos'] + ss['neg']))
        countNeg+=ss['neg']
        countNeu+=ss['neu']
        countPos+=ss['pos']
        
        print("============sentiment status============")
        print("compound="+str(df.at[date, 'compound']))
        print("neg="+str(df.at[date, 'neg']))
        print("neu="+str(df.at[date, 'neu']))
        print("pos="+str(df.at[date, 'pos']))
        print("ForGraph="+str(df.at[date, 'ForGraph']))
        #print("passed")
    except TypeError:
        #print("sentence is")
        #print(sentence)
        #print(df_vix.loc[date, 'articles'])
        print("ERROR")
        df.at[date, 'compound'] = 0
        df.at[date, 'neg'] = 0
        df.at[date, 'neu'] = 0
        df.at[date, 'pos'] = 0
#df.diff().hist(color="k", alpha=0.5, bins=50);



In [ ]:
#plt.figure();
#df['sentiment'] = df['ForGraph']#.ewm(span=20).mean()
#print(df.values)
senDF = df.values
senDF.all.hist(color="k", alpha=0.5, bins=50);

print("Total Neg="+str(countNeg)+" | Total Neu="+str(countNeu)+" | Total Pos="+str(countPos))

#df = pd.Series(data=df, index=pd.date_range("09/25/2019", periods=569))
# df['sentiment'] = df['ForGraph'].ewm(span=20).mean()
# df[['VIX']].plot(title='VIX by date')
#df.at[date, 'ForGraph'].plot(title='Sentiment by date')
#ts=pd.Series(df['VIX'],index=pd.date_range("25/09/2019", periods=569))
#df.diff().hist(color="k", alpha=0.5, bins=50);
#ts.plot()
#predictions_df_list['actual_value_ewm'] = predictions_df_list["actual_value"].ewm(span=20).mean()


In [ ]:
sentence = 'badly love'
from nltk.sentiment.vader import SentimentIntensityAnalyzer
import unicodedata
sid = SentimentIntensityAnalyzer()
sid
ss = sid.polarity_scores(sentence)
ss

In [ ]:
target_dist = sns.factorplot(x="date", data=df, kind="count", size=20, aspect=1.5, palette="PuBuGn_d")
plt.show();
target_dist.savefig('/Users/Farid/Documents/ProjectOptionsPredictions/Data/Graphs/target_dist_sentiment.png')

In [ ]:
# Checking date for empty strings
for date, row in df_vix.T.iteritems():
    if type(df_vix.loc[date, 'articles']).__name__ == 'str':
        print(date)

In [ ]:
df

In [ ]:
class TextCounts(BaseEstimator, TransformerMixin):
    
    def count_regex(self, pattern, tweet):
        return len(re.findall(pattern, tweet))
    
    def fit(self, X, y=None, **fit_params):
        # fit method is used when specific operations need to be done on the train data, but not on the test data
        return self
    
    def transform(self, X, **transform_params):
        count_words = X.apply(lambda x: self.count_regex(r'\w+', x)) 
        count_mentions = X.apply(lambda x: self.count_regex(r'@\w+', x))
        count_hashtags = X.apply(lambda x: self.count_regex(r'#\w+', x))
        count_capital_words = X.apply(lambda x: self.count_regex(r'\b[A-Z]{2,}\b', x))
        count_excl_quest_marks = X.apply(lambda x: self.count_regex(r'!|\?', x))
        count_urls = X.apply(lambda x: self.count_regex(r'http.?://[^\s]+[\s]?', x))
        # Moreover, it will result in having more words in the tweet
        count_emojis = X.apply(lambda x: emoji.demojize(x)).apply(lambda x: self.count_regex(r':[a-z_&]+:', x))
        
        df = pd.DataFrame({'count_words': count_words
                           , 'count_mentions': count_mentions
                           , 'count_hashtags': count_hashtags
                           , 'count_capital_words': count_capital_words
                           , 'count_excl_quest_marks': count_excl_quest_marks
                           , 'count_urls': count_urls
                           , 'count_emojis': count_emojis
                          })
        
        return df

In [ ]:
tc = TextCounts()
df_eda = tc.fit_transform(df_vix.articles)
# Add airline_sentiment to df_eda
df_eda['airline_sentiment'] = df.VIX

In [ ]:
def show_dist(df, col):
    print('Descriptive stats for {}'.format(col))
    print('-'*(len(col)+22))
    print(df.groupby('VIX')[col].describe())
    bins = np.arange(df[col].min(), df[col].max() + 1)
    g = sns.FacetGrid(df, col='VIX', size=5, hue='VIX', palette="PuBuGn_d")
    g = g.map(sns.distplot, col, kde=False, norm_hist=True, bins=bins)
    plt.show()

In [ ]:
target_dist = sns.factorplot(x="count_words", data=df, kind="count", size=6, aspect=1.5, palette="PuBuGn_d")
plt.show();
if not KAGGLE_ENV: target_dist.savefig('/Users/Farid/Documents/ProjectOptionsPredictions/Data/Graphs/target_dist.png')

In [ ]:
show_dist(df_vix, 'count_words')

In [ ]:
def show_dist(df, col):
    print('Descriptive stats for {}'.format(col))
    print('-'*(len(col)+22))
    print(df.groupby('airline_sentiment')[col].describe())
    bins = np.arange(df[col].min(), df[col].max() + 1)
    g = sns.FacetGrid(df, col='airline_sentiment', size=5, hue='airline_sentiment', palette="PuBuGn_d")
    g = g.map(sns.distplot, col, kde=False, norm_hist=True, bins=bins)
    plt.show()
    if not KAGGLE_ENV: g.savefig('../output/' + col + '_dist.png')

In [ ]:
train_start_date = '2019-09-26'
train_end_date = '2021-04-15'
test_start_date = '2019-09-26'
test_end_date = '2021-04-15'
#test_start_date = '2012-01-01'
#test_end_date = '2018-12-31'
train = df.loc[train_start_date : train_end_date]
print(test_start_date)
test = df.loc[test_start_date:test_end_date]
print(test_end_date)


In [ ]:
sentiment_score_list = []
for date, row in train.T.iteritems():
    #sentiment_score = np.asarray([df.loc[date, 'compound'],df.loc[date, 'neg'],df.loc[date, 'neu'],df.loc[date, 'pos']])
    sentiment_score = np.asarray([df.loc[date, 'neg'],df.loc[date, 'pos']])
    sentiment_score_list.append(sentiment_score)
numpy_df_train = np.asarray(sentiment_score_list)

In [ ]:
sentiment_score_list = []
for date, row in test.T.iteritems():
    #sentiment_score = np.asarray([df.loc[date, 'compound'],df.loc[date, 'neg'],df.loc[date, 'neu'],df.loc[date, 'pos']])
    sentiment_score = np.asarray([df.loc[date, 'neg'],df.loc[date, 'pos']])
    sentiment_score_list.append(sentiment_score)
numpy_df_test = np.asarray(sentiment_score_list)

In [ ]:
y_train = pd.DataFrame(train['VIX'])
y_test = pd.DataFrame(test['VIX'])

In [ ]:
from treeinterpreter import treeinterpreter as ti
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import classification_report,confusion_matrix

rf = RandomForestRegressor()
print("####################### 1 ######################")
print(y_train)
print("######################## 2 #####################")
print(numpy_df_train)
print("######################## 3 #####################")
rf.fit(numpy_df_train, y_train)

In [ ]:
print(rf.feature_importances_)

In [ ]:
prediction, bias, contributions = ti.predict(rf, numpy_df_test)

In [ ]:
prediction

In [ ]:
contributions

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
print(test_start_date)

idx = pd.date_range(test_start_date, test_end_date)
print(test_end_date)
predictions_df = pd.DataFrame(data=prediction[0:], index = idx, columns=['VIX'])

In [ ]:
predictions_df

In [ ]:
#predictions_df.plot() 
#test['VIX'].plot()

predictions_plot = predictions_df.plot()

fig = y_test.plot(ax = predictions_plot).get_figure()
fig.savefig("/Users/Farid/Documents/ProjectOptionsPredictions/Data/Graphs/random_forest_without_smoothing_sp500_financialmodelingprep.png")
#matplotlib.pyplot.savefig("C:\Users\Farid\Documents\Project Options predictions\graphs\random forest without smoothing.png")

In [ ]:
ax = predictions_df.rename(columns={"VIX": "predicted_VIX"}).plot(title='Random Forest predicted VIX 1 year')
ax.set_xlabel("Dates")
ax.set_ylabel("VIX")
fig = y_test.rename(columns={"VIX": "actual_VIX"}).plot(ax = ax).get_figure()
fig.savefig("/Users/Farid/Documents/ProjectOptionsPredictions/Data/Graphs/random_forest_without_smoothing2_sp500.png")

# colors = ['332288', '88CCEE', '44AA99', '117733', '999933', 'DDCC77', 'CC6677', '882255', 'AA4499']

In [ ]:
test

In [ ]:
# Increasing the VIX by a constant value so that it represents closing VIX during the testing

In [ ]:
from datetime import datetime, timedelta

temp_date = test_start_date
print(test_start_date)
print(datetime.astimezone)
average_last_5_days_test = 0
total_days = 10
for i in range(total_days):
    average_last_5_days_test += test.loc[temp_date, 'VIX']
    # Converting string to date time
    temp_date = datetime.strptime(temp_date, "%Y-%m-%d").date()
    # Reducing one day from date time
    difference = temp_date + timedelta(days=1)
    # Converting again date time to string
    temp_date = difference.strftime('%Y-%m-%d')
    #print temp_date
average_last_5_days_test = average_last_5_days_test / total_days
print (average_last_5_days_test)

temp_date = test_start_date
average_upcoming_5_days_predicted = 0
for i in range(total_days):
    average_upcoming_5_days_predicted += predictions_df.loc[temp_date, 'VIX']
    # Converting string to date time
    temp_date = datetime.strptime(temp_date, "%Y-%m-%d").date()
    # Adding one day from date time
    difference = temp_date + timedelta(days=1)
    # Converting again date time to string
    temp_date = difference.strftime('%Y-%m-%d')
    print(temp_date)
average_upcoming_5_days_predicted = average_upcoming_5_days_predicted / total_days
print (average_upcoming_5_days_predicted)
#average train.loc['2013-12-31', 'VIX'] - advpredictions_df.loc['2014-01-01', 'VIX']
difference_test_predicted_VIX = average_last_5_days_test - average_upcoming_5_days_predicted
print (difference_test_predicted_VIX)

In [ ]:
predictions_df

In [ ]:
# Adding 6177 to all the advpredictions_df VIX values
predictions_df['VIX'] = predictions_df['VIX'] + difference_test_predicted_VIX
predictions_df

In [ ]:
ax = predictions_df.rename(columns={"VIX": "predicted_VIX"}).plot(title='Random Forest predicted VIX 1 year after aligning')
ax.set_xlabel("Dates")
ax.set_ylabel("Options Volatility")
fig = y_test.rename(columns={"VIX": "actual_VIX"}).plot(ax = ax).get_figure()
fig.savefig("/Users/Farid/Documents/ProjectOptionsPredictions/Data/Graphs/random_forest_with_aligning_sp500.png")



# Smoothing the time series data

# Applying EWM pandas to smooth the VIX volatility

In [ ]:
predictions_df

In [ ]:
predictions_df['ewm'] = predictions_df["VIX"].ewm(span=60).mean()

In [ ]:
predictions_df

In [ ]:
predictions_df['actual_value'] = test['VIX']
predictions_df['actual_value_ewm'] = predictions_df["actual_value"].ewm(span=60).mean()

In [ ]:
predictions_df

In [ ]:
# Changing column names
predictions_df.columns = ['predicted_VIX', 'average_predicted_VIX', 'actual_VIX', 'average_actual_VIX']

In [ ]:
# Now plotting test predictions after smoothing
predictions_plot = predictions_df.plot(title='Random Forest predicted VIX 1 year after aligning & smoothing')
predictions_plot.set_xlabel("Dates")
predictions_plot.set_ylabel("VIX")
fig = predictions_plot.get_figure()
fig.savefig("/Users/Farid/Documents/ProjectOptionsPredictions/Data/Graphs/random_forest_after_smoothing_var_sp500.png")


In [ ]:
# Plotting just predict and actual average curves
predictions_df_average = predictions_df[['average_predicted_VIX', 'average_actual_VIX']]
predictions_plot = predictions_df_average.plot(title='Random Forest 1 years after aligning & smoothing')
predictions_plot.set_xlabel("Dates")
predictions_plot.set_ylabel("VIX")
fig = predictions_plot.get_figure()
fig.savefig("/Users/Farid/Documents/ProjectOptionsPredictions/Data/Graphs/random_forest_after_smoothing2_mean_sp500.png")

# Increasing the VIX by a constant value so that it represents closing VIX during the testing

In [ ]:
def offset_value(test_start_date, test, predictions_df):
    temp_date = test_start_date
    average_last_5_days_test = 0
    average_upcoming_5_days_predicted = 0
    total_days = 10
    for i in range(total_days):
        average_last_5_days_test += test.loc[temp_date, 'VIX']
        temp_date = datetime.strptime(temp_date, "%Y-%m-%d").date()
        difference = temp_date + timedelta(days=1)
        temp_date = difference.strftime('%Y-%m-%d')
    average_last_5_days_test = average_last_5_days_test / total_days

    temp_date = test_start_date
    for i in range(total_days):
        average_upcoming_5_days_predicted += predictions_df.loc[temp_date, 'VIX']
        temp_date = datetime.strptime(temp_date, "%Y-%m-%d").date()
        difference = temp_date + timedelta(days=1)
        temp_date = difference.strftime('%Y-%m-%d')
    average_upcoming_5_days_predicted = average_upcoming_5_days_predicted / total_days
    difference_test_predicted_VIX = average_last_5_days_test - average_upcoming_5_days_predicted
    return difference_test_predicted_VIX

# LogisticRegression

In [ ]:
from treeinterpreter import treeinterpreter as ti
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LogisticRegression
from datetime import datetime, timedelta

# average_upcoming_5_days_predicted += predictions_df.loc[temp_date, 'VIX']
# # Converting string to date time
# temp_date = datetime.strptime(temp_date, "%Y-%m-%d").date()
# # Adding one day from date time
# difference = temp_date + timedelta(days=1)
# # Converting again date time to string
# temp_date = difference.strftime('%Y-%m-%d')
        
# start_year = datetime.strptime(train_start_date, "%Y-%m-%d").date().month

prediction_list = []

# Splitting the training and testing data
train_start_date = '2019-09-25'
train_end_date = '2021-04-15'
test_start_date = '2019-09-25'
test_end_date = '2021-04-15'
train = df.loc[train_start_date : train_end_date]
test = df.loc[test_start_date : test_end_date]

# Calculating the sentiment score
sentiment_score_list = []
for date, row in train.T.iteritems():
    sentiment_score = np.asarray([df.loc[date, 'compound'],df.loc[date, 'neg'],df.loc[date, 'neu'],df.loc[date, 'pos']])
    #sentiment_score = np.asarray([df.loc[date, 'neg'],df.loc[date, 'pos']])
    sentiment_score_list.append(sentiment_score)
numpy_df_train = np.asarray(sentiment_score_list)
sentiment_score_list = []
print("#1")
for date, row in test.T.iteritems():
    sentiment_score = np.asarray([df.loc[date, 'compound'],df.loc[date, 'neg'],df.loc[date, 'neu'],df.loc[date, 'pos']])
    #sentiment_score = np.asarray([df.loc[date, 'neg'],df.loc[date, 'pos']])
    sentiment_score_list.append(sentiment_score)
numpy_df_test = np.asarray(sentiment_score_list)
# Generating models
# Linear regression model
lr = LogisticRegression()
print(sys.getsizeof(numpy_df_train))
print(sys.getsizeof(train['VIX']))

print("####################### 1 ######################")
print(train['VIX'])
print("######################## 2 #####################")
print(numpy_df_train)
print("######################## 3 #####################")
lr.fit(numpy_df_train, train['VIX'])

#   if len(np.sum(train['VIX'])) in [len(train['VIX']),0]:
#       print ("all one class")
#       #do something else
#   else:
#       #OK to proceed
#       lenreg.fit(numpy_df_train, train['VIX'])
#       y_pred = lenreg.predict(numpy_df_train)
#       print (metics.accuracy_score(y_test, y_pred))


print("#6")

prediction = lr.predict(numpy_df_test)
prediction_list.append(prediction)
print (train_start_date + ' ' + train_end_date + ' ' + test_start_date + ' ' + test_end_date)
idx = pd.date_range(test_start_date, test_end_date)
#print year
predictions_df_list = pd.DataFrame(data=prediction[0:], index = idx, columns=['VIX'])

difference_test_predicted_VIX = offset_value(test_start_date, test, predictions_df_list)
# Adding offset to all the advpredictions_df VIX values
predictions_df_list['VIX'] = predictions_df_list['VIX'] + difference_test_predicted_VIX
predictions_df_list

# Smoothing the plot
predictions_df_list['ewm'] = predictions_df_list["VIX"].ewm(span=10).mean()
predictions_df_list['actual_value'] = test['VIX']
predictions_df_list['actual_value_ewm'] = predictions_df_list["actual_value"].ewm(span=10).mean()
# Changing column names
predictions_df_list.columns = ['predicted_VIX', 'average_predicted_VIX', 'actual_VIX', 'average_actual_VIX']
predictions_df_list.plot()
predictions_df_list_average = predictions_df_list[['average_predicted_VIX', 'average_actual_VIX']]
predictions_df_list_average.plot()
print("#7")
#     predictions_df_list.show()


In [ ]:
lr.classes_

In [ ]:
lr.coef_[0]

# RandomForestRegressor

In [ ]:
from treeinterpreter import treeinterpreter as ti
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LogisticRegression
from datetime import datetime, timedelta

# average_upcoming_5_days_predicted += predictions_df.loc[temp_date, 'VIX']
# # Converting string to date time
# temp_date = datetime.strptime(temp_date, "%Y-%m-%d").date()
# # Adding one day from date time
# difference = temp_date + timedelta(days=1)
# # Converting again date time to string
# temp_date = difference.strftime('%Y-%m-%d')
        
# start_year = datetime.strptime(train_start_date, "%Y-%m-%d").date().month

prediction_list = []
# Splitting the training and testing data
train_start_date = '2019-09-25'
train_end_date = '2021-04-15'
test_start_date = '2020-09-25'
test_end_date = '2021-04-15'
train = df.loc[train_start_date : train_end_date]
test = df.loc[test_start_date:test_end_date]

# Calculating the sentiment score
sentiment_score_list = []
for date, row in train.T.iteritems():
    sentiment_score = np.asarray([df.loc[date, 'compound'],df.loc[date, 'neg'],df.loc[date, 'neu'],df.loc[date, 'pos']])
    #sentiment_score = np.asarray([df.loc[date, 'neg'],df.loc[date, 'pos']])
    sentiment_score_list.append(sentiment_score)
numpy_df_train = np.asarray(sentiment_score_list)
sentiment_score_list = []
for date, row in test.T.iteritems():
    sentiment_score = np.asarray([df.loc[date, 'compound'],df.loc[date, 'neg'],df.loc[date, 'neu'],df.loc[date, 'pos']])
    #sentiment_score = np.asarray([df.loc[date, 'neg'],df.loc[date, 'pos']])
    sentiment_score_list.append(sentiment_score)
numpy_df_test = np.asarray(sentiment_score_list)
# Generating models
rf = RandomForestRegressor(random_state=0)
rf.fit(numpy_df_train, train['VIX'])
#print rf

prediction, bias, contributions = ti.predict(rf, numpy_df_test)
prediction_list.append(prediction)
#print train_start_date + ' ' + train_end_date + ' ' + test_start_date + ' ' + test_end_date
idx = pd.date_range(test_start_date, test_end_date)
#print year
predictions_df_list = pd.DataFrame(data=prediction[0:], index = idx, columns=['VIX'])

difference_test_predicted_VIX = offset_value(test_start_date, test, predictions_df_list)
# Adding offset to all the advpredictions_df VIX values
predictions_df_list['VIX'] = predictions_df_list['VIX'] + difference_test_predicted_VIX
predictions_df_list

# Smoothing the plot
predictions_df_list['ewm'] = predictions_df_list["VIX"].ewm(span=60).mean()
predictions_df_list['actual_value'] = test['VIX']
predictions_df_list['actual_value_ewm'] = predictions_df_list["actual_value"].ewm(span=10).mean()
# Changing column names
predictions_df_list.columns = ['predicted_VIX', 'average_predicted_VIX', 'actual_VIX', 'average_actual_VIX']
predictions_df_list.plot()
predictions_df_list_average = predictions_df_list[['average_predicted_VIX', 'average_actual_VIX']]
predictions_df_list_average.plot()

#     predictions_df_list.show()


In [ ]:
# from IPython.display import Image  
# dot_data = tree.export_graphviz(rf, out_file=None, 
#                      feature_names=['comp', 'neg', 'neu', 'pos'],  
#                      class_names=iris.target_names,  
#                      filled=True, rounded=True,  
#                      special_characters=True)  
# graph = pydotplus.graph_from_dot_data(dot_data)  
# Image(graph.create_png())

# MLP Classifier

In [ ]:
from sklearn.neural_network import MLPClassifier
from datetime import datetime, timedelta

# average_upcoming_5_days_predicted += predictions_df.loc[temp_date, 'VIX']
# # Converting string to date time
# temp_date = datetime.strptime(temp_date, "%Y-%m-%d").date()
# # Adding one day from date time
# difference = temp_date + timedelta(days=1)
# # Converting again date time to string
# temp_date = difference.strftime('%Y-%m-%d')
        
# start_year = datetime.strptime(train_start_date, "%Y-%m-%d").date().month

prediction_list = []
# Splitting the training and testing data
train_start_date = '2019-09-25'
train_end_date = '2021-04-15'
test_start_date = '2019-09-25'
test_end_date = '2021-04-15'
train = df.loc[train_start_date : train_end_date]
test = df.loc[test_start_date:test_end_date]

# Calculating the sentiment score
sentiment_score_list = []
for date, row in train.T.iteritems():
    sentiment_score = np.asarray([df.loc[date, 'compound'],df.loc[date, 'neg'],df.loc[date, 'neu'],df.loc[date, 'pos']])
    #sentiment_score = np.asarray([df.loc[date, 'neg'],df.loc[date, 'pos']])
    sentiment_score_list.append(sentiment_score)
numpy_df_train = np.asarray(sentiment_score_list)
sentiment_score_list = []
for date, row in test.T.iteritems():
    sentiment_score = np.asarray([df.loc[date, 'compound'],df.loc[date, 'neg'],df.loc[date, 'neu'],df.loc[date, 'pos']])
    #sentiment_score = np.asarray([df.loc[date, 'neg'],df.loc[date, 'pos']])
    sentiment_score_list.append(sentiment_score)
numpy_df_test = np.asarray(sentiment_score_list)

# Generating models
mlpc = MLPClassifier(hidden_layer_sizes=(100, 200, 100), activation='relu', 
                        solver='lbfgs', alpha=0.005, learning_rate_init = 0.001, shuffle=False) # span = 20 # best 1
mlpc.fit(numpy_df_train, train['VIX'])   
prediction = mlpc.predict(numpy_df_test)

prediction_list.append(prediction)
#print train_start_date + ' ' + train_end_date + ' ' + test_start_date + ' ' + test_end_date
idx = pd.date_range(test_start_date, test_end_date)
#print year
predictions_df_list = pd.DataFrame(data=prediction[0:], index = idx, columns=['VIX'])

difference_test_predicted_VIX = offset_value(test_start_date, test, predictions_df_list)
# Adding offset to all the advpredictions_df VIX values
predictions_df_list['VIX'] = predictions_df_list['VIX'] + difference_test_predicted_VIX
predictions_df_list

# Smoothing the plot
predictions_df_list['ewm'] = predictions_df_list["VIX"].ewm(span=20).mean()
predictions_df_list['actual_value'] = test['VIX']
predictions_df_list['actual_value_ewm'] = predictions_df_list["actual_value"].ewm(span=20).mean()
# Changing column names
predictions_df_list.columns = ['predicted_VIX', 'average_predicted_VIX', 'actual_VIX', 'average_actual_VIX']
predictions_df_list.plot()
predictions_df_list_average = predictions_df_list[['average_predicted_VIX', 'average_actual_VIX']]
predictions_df_list_average.plot()

#     predictions_df_list.show()
    

In [ ]:

#my addition
import matplotlib.pyplot as plt
from sklearn.datasets import make_blobs

n_samples = 200
blob_centers = ([1, 1], [3, 4], [1, 3.3], [3.5, 1.8])
data, labels = make_blobs(n_samples=n_samples, 
                          centers=blob_centers, 
                          cluster_std=0.5,
                          random_state=0)


colours = ('green', 'orange', "blue", "magenta")
fig, ax = plt.subplots()

for n_class in range(len(blob_centers)):
    ax.scatter(data[labels==n_class][:, 0], 
               data[labels==n_class][:, 1], 
               c=colours[n_class], 
               s=30, 
               label=str(n_class))

In [ ]:
mlpc = MLPClassifier(hidden_layer_sizes=(100, 200, 100), activation='tanh', 
                         solver='lbfgs', alpha=0.010, learning_rate_init = 0.001, shuffle=False)
mlpc = MLPClassifier(hidden_layer_sizes=(100, 200, 100), activation='relu', 
                         solver='lbfgs', alpha=0.010, learning_rate_init = 0.001, shuffle=False) # span = 20
mlpc = MLPClassifier(hidden_layer_sizes=(100, 200, 100), activation='relu', 
                         solver='lbfgs', alpha=0.005, learning_rate_init = 0.001, shuffle=False) # span = 20 # best 1
mlpc = MLPClassifier(hidden_layer_sizes=(100, 200, 50), activation='relu', 
                         solver='lbfgs', alpha=0.005, learning_rate_init = 0.001, shuffle=False)
                     

In [ ]:
# checking the performance of training data itself
prediction, bias, contributions = ti.predict(rf, numpy_df_train)
idx = pd.date_range(train_start_date, train_end_date)
predictions_df1 = pd.DataFrame(data=prediction[0:], index = idx, columns=['VIX'])
predictions_df1.plot()
train['VIX'].plot()